In [1]:
from google.colab import drive
drive.mount('/content/drive')
!ls -lh /content/drive/MyDrive/hw1 #проверка, что колаб видит файлы

Mounted at /content/drive
total 5.3G
-rw------- 1 root root 1.2G Oct  3 04:27 oilMP_S4_L001_R1_001.fastq
-rw------- 1 root root 1.2G Oct  3 04:27 oilMP_S4_L001_R2_001.fastq
-rw------- 1 root root 1.6G Oct  3 04:27 oil_R1.fastq
-rw------- 1 root root 1.6G Oct  3 04:27 oil_R2.fastq
-rw------- 1 root root 2.1M Oct  3 04:05 platanus
-rw------- 1 root root 128K Oct  3 04:05 platanus_internal_trim
-rw------- 1 root root 128K Oct  3 04:05 platanus_trim


Для выполнения задания необходимо подготовить среду: создать папки, установить seqtk для случайной выборки чтений, fastQC для оценки качества чтений, multiQC для объединения результатов fastQC.

In [3]:
%%bash
#создадим папки для для данных и результата
mkdir -p /content/work/{sub,qcraw,qctrim,kk,fin,bb}
mkdir -p /content/drive/MyDrive/hw1/res
#установим программы для анализа (seqtk, fastQC, multiQC)
apt-get update
apt-get install -y seqtk fastqc
pip install multiqc
#проверка установки
fastqc --version
multiqc --version

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
seqtk is already the newest version (1.4-2).
fastqc is already the newest version (0.12.1+dfsg-3).
0 upgraded, 0 newly installed, 0 to remove and 62 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


Для сборки генома нам необходимы файлы программы Platanus. Предварительно загрузили эти файлы на Google Drive.

In [4]:
%%bash
#копируем программы Platanus в рабочую папку
mkdir -p /content/work/bb
cp /content/drive/MyDrive/hw1/platanus \
   /content/drive/MyDrive/hw1/platanus_trim \
   /content/drive/MyDrive/hw1/platanus_internal_trim \
   /content/work/bb/
#делаем программы исполняемыми
chmod +x /content/work/bb/*

In [5]:
import os
#путь к программам Platanus
os.environ["PATH"] += ":/content/work/bb"
SEED = 412
RAW = "/content/drive/MyDrive/hw1"
W = "/content/work"

Создаем случайные уменьшенные подвыборки FASTQ, из обычных парных чтений случайно выберем по 5 миллионов ридов, а из mate-pair –– 1,5 миллионов.

In [6]:
%%bash -s "$RAW" "$W" "$SEED"
RAW=$1
W=$2
SEED=$3
cd $W/sub
#берем случайную подвыборку чтений
seqtk sample -s$SEED $RAW/oil_R1.fastq 5000000 > oil_R1.sub.fastq
seqtk sample -s$SEED $RAW/oil_R2.fastq 5000000 > oil_R2.sub.fastq
seqtk sample -s$SEED $RAW/oilMP_S4_L001_R1_001.fastq 1500000 > oilMP_R1.sub.fastq
seqtk sample -s$SEED $RAW/oilMP_S4_L001_R2_001.fastq 1500000 > oilMP_R2.sub.fastq
#проверяем количество строк
wc -l *.fastq

   6000000 oilMP_R1.sub.fastq
   6000000 oilMP_R2.sub.fastq
  20000000 oil_R1.sub.fastq
  20000000 oil_R2.sub.fastq
  52000000 total


Проверим качество полученных файлов с помощью FastQC и объеденим результаты с помощью MultiQC. Проверяем качество чтений по позициям, GC-состав, количество дубликатов и наличие адаптеров.

In [7]:
%%bash -s "$W"
W=$1
cd $W
fastqc -t 1 -o qcraw sub/*.fastq
multiqc qcraw -o qcraw -n multiqc_raw
cp -r qcraw /content/drive/MyDrive/hw1/res/

[0.003s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.003s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
null
null
null
null
Analysis complete for oilMP_R1.sub.fastq
Analysis complete for oilMP_R2.sub.fastq
Analysis complete for oil_R1.sub.fastq
Analysis complete for oil_R2.sub.fastq


Started analysis of oilMP_R1.sub.fastq
Approx 5% complete for oilMP_R1.sub.fastq
Approx 10% complete for oilMP_R1.sub.fastq
Approx 15% complete for oilMP_R1.sub.fastq
Approx 20% complete for oilMP_R1.sub.fastq
Approx 25% complete for oilMP_R1.sub.fastq
Approx 30% complete for oilMP_R1.sub.fastq
Approx 35% complete for oilMP_R1.sub.fastq
Approx 40% complete for oilMP_R1.sub.fastq
Approx 45% complete for oilMP_R1.sub.fastq
Approx 50% complete for oilMP_R1.sub.fastq
Approx 55% complete for oilMP_R1.sub.fastq
Approx 60% complete for oilMP_R1.sub.fastq
Approx 65% complete for oilMP_R1.sub.fastq
Approx 70% complete for oilMP_R1.sub.fastq
Approx 75% complete for oilMP_R1.sub.fastq
Approx 80% complete for oilMP_R1.sub.fastq
Approx 85% complete for oilMP_R1.sub.fastq
Approx 90% complete for oilMP_R1.sub.fastq
Approx 95% complete for oilMP_R1.sub.fastq
Approx 100% complete for oilMP_R1.sub.fastq
Started analysis of oilMP_R2.sub.fastq
Approx 5% complete for oilMP_R2.sub.fastq
Approx 10% complete 

Удалим адаптеры и низкокачественные участки чтений. Для обычных чтений используем platanus trim, для mate-pair –– platanus internal trim, тк адаптер в них может находиться внутри последовательности.

In [9]:
%%bash -s "$W"
W=$1
cd $W/sub
platanus_trim oil_R1.sub.fastq oil_R2.sub.fastq
platanus_internal_trim oilMP_R1.sub.fastq oilMP_R2.sub.fastq
ls -lh #проверим файлы

total 7.4G
-rw-r--r-- 1 root root 801M Oct  4 18:17 oilMP_R1.sub.fastq
-rw-r--r-- 1 root root 271M Oct  4 19:01 oilMP_R1.sub.fastq.int_trimmed
-rw-r--r-- 1 root root 801M Oct  4 18:17 oilMP_R2.sub.fastq
-rw-r--r-- 1 root root 261M Oct  4 19:01 oilMP_R2.sub.fastq.int_trimmed
-rw-r--r-- 1 root root 1.3G Oct  4 18:16 oil_R1.sub.fastq
-rw-r--r-- 1 root root 1.5G Oct  4 18:58 oil_R1.sub.fastq.trimmed
-rw-r--r-- 1 root root 1.3G Oct  4 18:16 oil_R2.sub.fastq
-rw-r--r-- 1 root root 1.4G Oct  4 18:58 oil_R2.sub.fastq.trimmed


Running with trim adapter mode
Checking files: 
  oil_R1.sub.fastq oil_R2.sub.fastq  (100%)

Number of trimmed read with adapter: 
NUM_OF_TRIMMED_READ(FORWARD) = 209631
NUM_OF_TRIMMED_BASE(FORWARD) = 208805
NUM_OF_TRIMMED_READ(REVERSE) = 209679
NUM_OF_TRIMMED_BASE(REVERSE) = 353941
NUM_OF_TRIMMED_PAIR(OR) = 209705
NUM_OF_TRIMMED_PAIR(AND) = 209605


Number of trimmed read because of low quality or too short (< 11bp): 
NUM_OF_TRIMMED_READ(FORWARD) = 908053
NUM_OF_TRIMMED_BASE(FORWARD) = 18128206
NUM_OF_TRIMMED_READ(REVERSE) = 1178791
NUM_OF_TRIMMED_BASE(REVERSE) = 35888540
NUM_OF_TRIMMED_PAIR(OR) = 1630245
NUM_OF_TRIMMED_PAIR(AND) = 456599


#### PROCESS INFORMATION ####
User Time:           2.71 min
System Time:         0.25 min
VmPeak:           0.116 GByte
VmHWM:            0.110 GByte
Execution time:      3.07 min

Running with trim internal adapter mode
Checking files: 
  oilMP_R1.sub.fastq oilMP_R2.sub.fastq  (100%)

Number of trimmed read with internal adapter: 
NUM_OF_TRIMMED_RE

Повторно проверяем качество чтений и сравним результат QC до очистки.

In [10]:
%%bash -s "$W"
W=$1
cd $W
fastqc -t 1 -o qctrim sub/*.trimmed sub/*.int_trimmed
multiqc qctrim -o qctrim -n multiqc_trim
cp -r qctrim /content/drive/MyDrive/hw1/res/

[0.004s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.004s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
null
null
null
null
Analysis complete for oil_R1.sub.fastq.trimmed
Analysis complete for oil_R2.sub.fastq.trimmed
Analysis complete for oilMP_R1.sub.fastq.int_trimmed
Analysis complete for oilMP_R2.sub.fastq.int_trimmed


Started analysis of oil_R1.sub.fastq.trimmed
Approx 5% complete for oil_R1.sub.fastq.trimmed
Approx 10% complete for oil_R1.sub.fastq.trimmed
Approx 15% complete for oil_R1.sub.fastq.trimmed
Approx 20% complete for oil_R1.sub.fastq.trimmed
Approx 25% complete for oil_R1.sub.fastq.trimmed
Approx 30% complete for oil_R1.sub.fastq.trimmed
Approx 35% complete for oil_R1.sub.fastq.trimmed
Approx 40% complete for oil_R1.sub.fastq.trimmed
Approx 45% complete for oil_R1.sub.fastq.trimmed
Approx 50% complete for oil_R1.sub.fastq.trimmed
Approx 55% complete for oil_R1.sub.fastq.trimmed
Approx 60% complete for oil_R1.sub.fastq.trimmed
Approx 65% complete for oil_R1.sub.fastq.trimmed
Approx 70% complete for oil_R1.sub.fastq.trimmed
Approx 75% complete for oil_R1.sub.fastq.trimmed
Approx 80% complete for oil_R1.sub.fastq.trimmed
Approx 85% complete for oil_R1.sub.fastq.trimmed
Approx 90% complete for oil_R1.sub.fastq.trimmed
Approx 95% complete for oil_R1.sub.fastq.trimmed
Started analysis of oil_R

Из очищенных парных чтений собираем контиги (более длинные последовательости). Platanus assemble строит сборку на основе перекрывающихся участков.

In [13]:
%%bash -s "$W"
W=$1
cd $W/kk
platanus assemble -o Oil -f ../sub/oil_R1.sub.fastq.trimmed ../sub/oil_R2.sub.fastq.trimmed \
  -t 1 -m 8 2> assemble.log #ограничили память до 8 гб
tail -5 assemble.log #выведем последние 5 строк посмотреть сборку
ls -lh
cp Oil_* assemble.log /content/drive/MyDrive/hw1/res/

assemble completed!

#### PROCESS INFORMATION ####
VmPeak:           7.189 GByte
VmHWM:            7.186 GByte
total 3.9M
-rw-r--r-- 1 root root 4.9K Oct  4 19:54 assemble.log
-rw-r--r-- 1 root root  53K Oct  4 19:32 Oil_32merFrq.tsv
-rw-r--r-- 1 root root    0 Oct  4 19:53 Oil_contigBubble.fa
-rw-r--r-- 1 root root 3.9M Oct  4 19:54 Oil_contig.fa


Соединяем отдельные контиги в скаффолды (более длинные). PE помогают соединить близко расположенные контиги, MP связывают контиги на большом расстоянии.

In [14]:
%%bash -s "$W"
W=$1
cd $W/kk
platanus scaffold -o Oil -c Oil_contig.fa -b Oil_contigBubble.fa \
  -IP1 ../sub/oil_R1.sub.fastq.trimmed ../sub/oil_R2.sub.fastq.trimmed \
  -OP2 ../sub/oilMP_R1.sub.fastq.int_trimmed ../sub/oilMP_R2.sub.fastq.int_trimmed \
  -t 1 2> scaffold.log
ls -lh
cp Oil_scaffold* scaffold.log /content/drive/MyDrive/hw1/res/

total 12M
-rw-r--r-- 1 root root 4.9K Oct  4 19:54 assemble.log
-rw-r--r-- 1 root root  53K Oct  4 19:32 Oil_32merFrq.tsv
-rw-r--r-- 1 root root    0 Oct  4 19:53 Oil_contigBubble.fa
-rw-r--r-- 1 root root 3.9M Oct  4 19:54 Oil_contig.fa
-rw-r--r-- 1 root root 958K Oct  4 20:00 Oil_lib1_insFreq.tsv
-rw-r--r-- 1 root root 3.0M Oct  4 20:00 Oil_lib2_insFreq.tsv
-rw-r--r-- 1 root root    0 Oct  4 20:01 Oil_scaffoldBubble.fa
-rw-r--r-- 1 root root 3.4K Oct  4 20:01 Oil_scaffoldComponent.tsv
-rw-r--r-- 1 root root 3.8M Oct  4 20:01 Oil_scaffold.fa
-rw-r--r-- 1 root root  11K Oct  4 20:01 scaffold.log


После построения скаффолдов в них могут остаться N –– участки с неизвестной последовательностью, platanus gap close заполняет эти участки с помощью исходных чтений.

In [15]:
%%bash -s "$W"
W=$1
cd $W/kk
platanus gap_close -o Oil -c Oil_scaffold.fa \
  -IP1 ../sub/oil_R1.sub.fastq.trimmed ../sub/oil_R2.sub.fastq.trimmed \
  -OP2 ../sub/oilMP_R1.sub.fastq.int_trimmed ../sub/oilMP_R2.sub.fastq.int_trimmed \
  -t 1 2> gapclose.log
ls -lh
cp Oil_gapClosed.fa gapclose.log /content/drive/MyDrive/hw1/res/

total 16M
-rw-r--r-- 1 root root 4.9K Oct  4 19:54 assemble.log
-rw-r--r-- 1 root root 1.9K Oct  4 20:09 gapclose.log
-rw-r--r-- 1 root root  53K Oct  4 19:32 Oil_32merFrq.tsv
-rw-r--r-- 1 root root    0 Oct  4 19:53 Oil_contigBubble.fa
-rw-r--r-- 1 root root 3.9M Oct  4 19:54 Oil_contig.fa
-rw-r--r-- 1 root root 3.8M Oct  4 20:09 Oil_gapClosed.fa
-rw-r--r-- 1 root root 958K Oct  4 20:00 Oil_lib1_insFreq.tsv
-rw-r--r-- 1 root root 3.0M Oct  4 20:00 Oil_lib2_insFreq.tsv
-rw-r--r-- 1 root root    0 Oct  4 20:01 Oil_scaffoldBubble.fa
-rw-r--r-- 1 root root 3.4K Oct  4 20:01 Oil_scaffoldComponent.tsv
-rw-r--r-- 1 root root 3.8M Oct  4 20:01 Oil_scaffold.fa
-rw-r--r-- 1 root root  11K Oct  4 20:01 scaffold.log


Статистика контигов и скаффолдов. Рассчитаем характеристики сборки (кол-во последовательностей, общую длину, длину самой последовательности, N50 и N).

In [16]:
import re
#читаем фаста-файлы и сохраняем в виде словаря
def read_fasta(file):
    seqs = {}
    n = ""
    with open(file) as f:
        for line in f:
            line = line.strip()

            if line.startswith(">"):
                n = line[1:].split()[0]
                seqs[n] = ""
            else:
                seqs[n] += line.upper()
    return seqs

#считаем N50
def n50(sequences):
    lengths = [len(seq) for seq in sequences.values()]
    lengths.sort(reverse=True)
    half = sum(lengths) / 2
    total = 0
    for length in lengths:
        total += length
        if total >= half:
            return length

#считаем характеристики сборки
def stats(file):
    seqs = read_fasta(file)
    lengths = [len(seq) for seq in seqs.values()]
    print("Файл:", file)
    print("Колво последовательностей:", len(lengths))
    print("Общая длина:", sum(lengths))
    print("Самая длинная последовательность:", max(lengths))
    print("N50:", n50(seqs))
    print("Колво N:", sum(seq.count("N") for seq in seqs.values()))
    return seqs

#гэпы в самой длинной последовательности
def find_gaps(seqs):
    # Находим самую длинную последовательность
    n = max(seqs, key=lambda x: len(seqs[x]))
    seq = seqs[n]
    # Ищем участки, состоящие из N
    gaps = re.findall(r"N+", seq)
    print("Самая длинная последовательность:", n)
    print("Ее длина:", len(seq))
    print("Колво гэпов:", len(gaps))
    print("Общая длина гэпов:", sum(len(gap) for gap in gaps))
    return n

D = "/content/work/kk/"
contigs = stats(D + "Oil_contig.fa")
scaffolds = stats(D + "Oil_scaffold.fa")
closed = stats(D + "Oil_gapClosed.fa")

Файл: /content/work/kk/Oil_contig.fa
Колво последовательностей: 598
Общая длина: 3924310
Самая длинная последовательность: 135083
N50: 47993
Колво N: 0
Файл: /content/work/kk/Oil_scaffold.fa
Колво последовательностей: 69
Общая длина: 3874071
Самая длинная последовательность: 3834604
N50: 3834604
Колво N: 6835
Файл: /content/work/kk/Oil_gapClosed.fa
Колво последовательностей: 69
Общая длина: 3910458
Самая длинная последовательность: 3870738
N50: 3870738
Колво N: 1904


Бонусное задание. Для бонуса потворили те же этапы сборки, но использовали меньшее количество чтений.

In [17]:
#сборка
TAG  = "half"
#количество чтений
N_PE = 2500000
N_MP = 750000

In [18]:
%%bash -s "$RAW" "$W" "$SEED" "$TAG" "$N_PE" "$N_MP"
RAW=$1
W=$2
SEED=$3
TAG=$4
N_PE=$5
N_MP=$6
#Создадим отдельную папку для бонусной сборки
mkdir -p $W/bonus/$TAG
cd $W/bonus/$TAG

#Возьмем меньшую подвыборку из исходных файлов
seqtk sample -s$SEED $RAW/oil_R1.fastq $N_PE > pe_R1.fastq
seqtk sample -s$SEED $RAW/oil_R2.fastq $N_PE > pe_R2.fastq
seqtk sample -s$SEED $RAW/oilMP_S4_L001_R1_001.fastq $N_MP > mp_R1.fastq
seqtk sample -s$SEED $RAW/oilMP_S4_L001_R2_001.fastq $N_MP > mp_R2.fastq

#Проверим количество строк в полученных файлах
wc -l *.fastq
#Тримминг
platanus_trim pe_R1.fastq pe_R2.fastq
platanus_internal_trim mp_R1.fastq mp_R2.fastq
#удалим исходные подвыборки
rm pe_R1.fastq pe_R2.fastq mp_R1.fastq mp_R2.fastq
#сборка контигов
platanus assemble -o $TAG \
  -f pe_R1.fastq.trimmed pe_R2.fastq.trimmed \
  -t 1 -m 8 2> assemble.log
#скаффолды
platanus scaffold -o $TAG \
  -c ${TAG}_contig.fa \
  -b ${TAG}_contigBubble.fa \
  -IP1 pe_R1.fastq.trimmed pe_R2.fastq.trimmed \
  -OP2 mp_R1.fastq.int_trimmed mp_R2.fastq.int_trimmed \
  -t 1 2> scaffold.log
#закрытие гэпов
platanus gap_close -o $TAG \
  -c ${TAG}_scaffold.fa \
  -IP1 pe_R1.fastq.trimmed pe_R2.fastq.trimmed \
  -OP2 mp_R1.fastq.int_trimmed mp_R2.fastq.int_trimmed \
  -t 1 2> gapclose.log
#удалим временные файлы и сохраним результаты
rm *.trimmed *.int_trimmed
mkdir -p /content/drive/MyDrive/hw1/res/bonus_$TAG
cp ${TAG}_contig.fa ${TAG}_scaffold.fa ${TAG}_gapClosed.fa \
   *.log /content/drive/MyDrive/hw1/res/bonus_$TAG/
ls -lh

   3000000 mp_R1.fastq
   3000000 mp_R2.fastq
  10000000 pe_R1.fastq
  10000000 pe_R2.fastq
  26000000 total
total 16M
-rw-r--r-- 1 root root 4.3K Oct  4 20:48 assemble.log
-rw-r--r-- 1 root root 1.9K Oct  4 20:52 gapclose.log
-rw-r--r-- 1 root root  27K Oct  4 20:35 half_32merFrq.tsv
-rw-r--r-- 1 root root  400 Oct  4 20:47 half_contigBubble.fa
-rw-r--r-- 1 root root 3.9M Oct  4 20:48 half_contig.fa
-rw-r--r-- 1 root root 3.8M Oct  4 20:52 half_gapClosed.fa
-rw-r--r-- 1 root root 1.5M Oct  4 20:49 half_lib1_insFreq.tsv
-rw-r--r-- 1 root root 2.6M Oct  4 20:49 half_lib2_insFreq.tsv
-rw-r--r-- 1 root root    0 Oct  4 20:49 half_scaffoldBubble.fa
-rw-r--r-- 1 root root 3.5K Oct  4 20:49 half_scaffoldComponent.tsv
-rw-r--r-- 1 root root 3.8M Oct  4 20:49 half_scaffold.fa
-rw-r--r-- 1 root root  11K Oct  4 20:49 scaffold.log


Running with trim adapter mode
Checking files: 
  pe_R1.fastq pe_R2.fastq  (100%)

Number of trimmed read with adapter: 
NUM_OF_TRIMMED_READ(FORWARD) = 104462
NUM_OF_TRIMMED_BASE(FORWARD) = 104801
NUM_OF_TRIMMED_READ(REVERSE) = 104494
NUM_OF_TRIMMED_BASE(REVERSE) = 179347
NUM_OF_TRIMMED_PAIR(OR) = 104510
NUM_OF_TRIMMED_PAIR(AND) = 104446


Number of trimmed read because of low quality or too short (< 11bp): 
NUM_OF_TRIMMED_READ(FORWARD) = 453650
NUM_OF_TRIMMED_BASE(FORWARD) = 9043398
NUM_OF_TRIMMED_READ(REVERSE) = 589041
NUM_OF_TRIMMED_BASE(REVERSE) = 17909324
NUM_OF_TRIMMED_PAIR(OR) = 814528
NUM_OF_TRIMMED_PAIR(AND) = 228163


#### PROCESS INFORMATION ####
User Time:           1.23 min
System Time:         0.05 min
VmPeak:           0.116 GByte
VmHWM:            0.109 GByte
Execution time:      1.32 min

Running with trim internal adapter mode
Checking files: 
  mp_R1.fastq mp_R2.fastq  (100%)

Number of trimmed read with internal adapter: 
NUM_OF_TRIMMED_READ(FORWARD) = 486083
NUM_OF

In [23]:
import re
import pandas as pd
#считаем показатели для одной сборки
def metrics(file):
    seqs = read_fasta(file)
    l = [len(seq) for seq in seqs.values()]
    #находим самую длинную последовательность
    lo = max(seqs.values(), key=len)
    #ищем участки из N
    gaps = re.findall(r"N+", lo)
    return {
        "число": len(l),
        "общая длина": sum(l),
        "самая длинная": max(l),
        "N50": n50(l),
        "число гэпов": len(gaps),
        "длина гэпов": sum(len(gap) for gap in gaps)
    }
#считаем N50
def n50(lengths):
    lengths = sorted(lengths, reverse=True)
    half = sum(lengths) / 2
    total = 0
    for length in lengths:
        total += length
        if total >= half:
            return length
run = {
    "5M PE + 1.5M MP": "/content/work/kk/Oil",
    "2.5M PE + 0.75M MP": "/content/work/bonus/half/half"
}
#результаты
r = {}
for n, path in run.items():
    con = metrics(path + "_contig.fa")
    sca = metrics(path + "_scaffold.fa")
    cl = metrics(path + "_gapClosed.fa")
    r[n] = {
        "Контигов": con["число"],
        "N50 контигов": con["N50"],
        "Самый длинный контиг": con["самая длинная"],
        "Скаффолдов": sca["число"],
        "N50 скаффолдов": sca["N50"],
        "Самый длинный скаффолд": sca["самая длинная"],
        "Гэпов до gap_close": sca["число гэпов"],
        "Гэпов после gap_close": cl["число гэпов"],
        "Общая длина сборки": cl["общая длина"]
    }
tab = pd.DataFrame(r)
display(tab)

,5M PE + 1.5M MP,2.5M PE + 0.75M MP
Контигов,598,707
N50 контигов,47993,81139
Самый длинный контиг,135083,235073
Скаффолдов,69,80
N50 скаффолдов,3834604,3809259
Самый длинный скаффолд,3834604,3809259
Гэпов до gap_close,60,71
Гэпов после gap_close,8,14
Общая длина сборки,3910458,3873833
